# 03 Work Trips

Build rush-hour work-trip datasets for Streamlit map exploration.

Filters in this notebook:
- Weekdays only (Monday-Friday)
- Morning: 06-09
- Evening: 14-17

Exports:
- work_trips_filtered.csv
- work_trips_routes.csv

In [5]:
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from utils import load_app_ready, export_df

In [6]:
df = load_app_ready()
df.shape

(15907082, 22)

In [7]:
required = [
    'city_name', 'year', 'trip_id', 'start_hour', 'duration_seconds',
    'day_of_week', 'day_name',
    'start_station_name', 'start_lat', 'start_lon',
    'end_station_name', 'end_lat', 'end_lon'
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns for work-trip analysis: {missing}')

base = df.dropna(
    subset=['city_name', 'year', 'trip_id', 'start_hour', 'duration_seconds', 'day_of_week', 'day_name']
).copy()
base['year'] = pd.to_numeric(base['year'], errors='coerce')
base['start_hour'] = pd.to_numeric(base['start_hour'], errors='coerce')
base['duration_seconds'] = pd.to_numeric(base['duration_seconds'], errors='coerce')
base['day_of_week'] = pd.to_numeric(base['day_of_week'], errors='coerce')
base = base.dropna(subset=['year', 'start_hour', 'duration_seconds', 'day_of_week'])
base['year'] = base['year'].astype(int)
base['start_hour'] = base['start_hour'].astype(int)
base['day_of_week'] = base['day_of_week'].astype(int)
base['duration_minutes'] = (base['duration_seconds'] / 60).round(2)

# Keep weekdays only (Mon-Fri).
base = base[base['day_of_week'].between(0, 4)].copy()

is_morning = base['start_hour'].between(6, 9)
is_evening = base['start_hour'].between(14, 17)
work_trips = base[is_morning | is_evening].copy()
work_trips['time_bin'] = np.where(
    work_trips['start_hour'].between(6, 9),
    'Morning (06-09)',
    'Evening (14-17)'
)

work_trips = work_trips.dropna(
    subset=['start_station_name', 'end_station_name', 'start_lat', 'start_lon', 'end_lat', 'end_lon']
)
work_trips = work_trips[
    work_trips['start_station_name'].astype(str).str.strip().str.lower()
    != work_trips['end_station_name'].astype(str).str.strip().str.lower()
]

duration_bins = [0, 5, 10, 15, 20, 25, 30, float('inf')]
duration_labels = [
    '0-5 min',
    '6-10 min',
    '11-15 min',
    '16-20 min',
    '21-25 min',
    '25-30 min',
    '30+ min',
]
work_trips['duration_bin'] = pd.cut(
    work_trips['duration_minutes'],
    bins=duration_bins,
    labels=duration_labels,
    include_lowest=True,
    right=True,
)
work_trips = work_trips.dropna(subset=['duration_bin']).copy()
work_trips['duration_bin'] = work_trips['duration_bin'].astype(str)

work_routes = (
    work_trips.groupby(
        [
            'city_name', 'year', 'day_of_week', 'day_name', 'time_bin', 'duration_bin',
            'start_station_name', 'start_lat', 'start_lon',
            'end_station_name', 'end_lat', 'end_lon',
        ],
        as_index=False,
    )
    .agg(
        trips=('trip_id', 'count'),
        avg_duration_minutes=('duration_minutes', 'mean'),
        median_duration_minutes=('duration_minutes', 'median'),
    )
    .sort_values(
        ['city_name', 'year', 'day_of_week', 'time_bin', 'duration_bin', 'trips'],
        ascending=[True, True, True, True, True, False],
    )
)

work_routes['avg_duration_minutes'] = work_routes['avg_duration_minutes'].round(2)
work_routes['median_duration_minutes'] = work_routes['median_duration_minutes'].round(2)

work_trips.shape, work_routes.shape

((6806839, 25), (3047887, 15))

In [8]:
export_df('work_trips_filtered', work_trips)
export_df('work_trips_routes', work_routes)
print('Exported work trip tables.')

[bridge] Exported DataFrame → c:\Users\matia\Desktop\Projects\bysykkel\Urban-Cycling\02_data\gold\notebook_exports\work_trips_filtered.csv
[bridge] Exported DataFrame → c:\Users\matia\Desktop\Projects\bysykkel\Urban-Cycling\02_data\gold\notebook_exports\work_trips_routes.csv
Exported work trip tables.
